# EDA (Exploratory Data Analysis) of the dataset

Exploration of the Abalone dataset with `ydata-profiling`. The goal is to predict the number of `Rings` (age = Rings + 1.5 years) from physical measurements.

# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
from ydata_profiling import ProfileReport

pd.set_option("display.max_columns", 500)

# Data

In [ ]:
df = pd.read_csv("../data/abalone.csv")
df.head()

# EDA

In [ ]:
profile = ProfileReport(df, title="Abalone Dataset Profiling")
profile.to_notebook_iframe()

In [ ]:
# ydata-profiling switches matplotlib to a non-display backend, so we switch it back
%matplotlib inline

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, feature in zip(axes, ["Whole weight", "Diameter"]):
    ax.scatter(df[feature], df["Height"], s=10, alpha=0.4)
    ax.set_xlabel(feature)
    ax.set_ylabel("Height")
    ax.set_title(f"Height vs {feature}")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Conclusions

- **Clean data**: 4,177 rows, 9 columns, no missing values and no duplicates.
- **Target**: `Rings` goes from 1 to 29 (mean 9.9, median 9) and is right-skewed: most abalones have 8 to 11 rings, old ones are rare.
- **Outliers**: 2 rows have `Height = 0` (impossible) and 2 have `Height > 0.4` (vs. a 0.14 mean). These are likely measurement errors.
- **Strong multicollinearity**: all size and weight features are highly correlated with each other (0.8 to 0.99, e.g. `Length` vs `Diameter` = 0.99).
- **Predictors of `Rings`**: every numerical feature is positively correlated with `Rings`, `Shell weight` being the best one (0.63). No single feature is enough to predict the age precisely.
- **`Sex` is useful**: the 3 classes are balanced (M 1,528, I 1,342, F 1,307) and infants (`I`) have far fewer rings (7.9 on average vs. 10.7 for M and 11.1 for F). It must be one-hot encoded for the model.

## Why we remove 4 `Height` outliers

We remove the 2 abalones with `Height > 0.4` and the 2 with `Height = 0`, which leaves 4,173 rows (0.1% of the data removed).

**`Height > 0.4`** (`Height` = 1.13 and 0.515): they are far away from the rest of the dataset.

- **Compared to the other abalones** (mean height 0.139, std 0.038, max 0.25), they are **25.7 and 9.8 standard deviations** above the mean, and **4.5 and 2.1 times taller** than the tallest remaining abalone.
- **Compared to abalones of the same diameter**, they don't make physical sense: the one with `Height` = 1.13 has a diameter of 0.355, whereas other abalones of that diameter are between 0.08 and 0.18 high. It would be **3 times higher than wide**, while abalones are flat shells (height ≈ 0.34 × diameter on average).
- **Their other measurements are normal** (weights, length, rings), so only `Height` is wrong: these are most likely typos (e.g. 1.13 instead of 0.113).

**`Height = 0`**: a shell cannot have a zero height, and every other abalone is at least 0.01 high. For their diameters (0.34 and 0.23), the other abalones are between 0.09 and 0.18, and between 0.05 and 0.105 high. These are missing values recorded as 0. One of them is also inconsistent elsewhere: its `Shell weight` (0.35) is higher than its `Whole weight` (0.13).

Removing them loses no information, while keeping them would distort a linear model (they stretch the `Height` scale and pull the regression line).